In [1]:
from imodels.tree.rf_plus.rf_plus.rf_plus_models import RandomForestPlusRegressor
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
from imodels.tree.rf_plus.rf_plus.rf_plus_models import RandomForestPlusRegressor, RandomForestPlusClassifier
from sklearn.ensemble import RandomForestRegressor
from imodels.tree.rf_plus.feature_importance.rfplus_explainer import *

/scratch/users/zachrewolinski/conda/envs/mdi/lib/python3.10/site-packages/glmnet/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/scratch/users/zachrewolinski/conda/envs/mdi/lib/python3.10/site-packages/pkg_resources/__init__.py:3146: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/scratch/users/zachrewolinski/conda/envs/mdi/lib/python3.10/site-packages/pkg_resources/__init__.py:3146: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits.basemap_data')`

In [2]:
X_361254 = pd.read_csv("361254/X.csv")
y_361254 = pd.read_csv("361254/y.csv")

X_361254 = X_361254.sample(n=100, random_state=42).values
y_361254 = y_361254.sample(n=100, random_state=42).values

# choose first two variables for simplicity
X_361254 = X_361254[:, :2]

In [3]:
rf = RandomForestRegressor(n_estimators=1, random_state=42, max_depth=2)
rf.fit(X_361254, y_361254.ravel())

rf_plus_inbag = RandomForestPlusRegressor(rf_model=rf, include_raw=False, fit_on="inbag", prediction_model=LinearRegression())
rf_plus_inbag.fit(X_361254, y_361254.ravel(), n_jobs=None)

In [4]:
rf.estimators_[0].get_depth()

2

In [5]:
rf.estimators_[0].tree_.impurity

array([319.27671871, 352.38738538, 212.05457729, 297.47790602,
       200.43355437, 183.52534018,   0.        ])

In [6]:
tree = rf.estimators_[0].tree_
n_nodes = tree.node_count

children_left = tree.children_left
children_right = tree.children_right
samples = tree.n_node_samples
features = tree.feature

# Step 3: Identify leaf nodes
is_leaf = (children_left == -1) & (children_right == -1)
leaf_ids = [i for i in range(n_nodes) if is_leaf[i]]

# Step 4: Build parent links and directions
parent = {0: None}  # root has no parent
direction = {}      # child node -> "left" or "right"
stack = [0]

while stack:
    node_id = stack.pop()
    left = children_left[node_id]
    right = children_right[node_id]
    if left != -1:
        parent[left] = node_id
        direction[left] = "left"
        stack.append(left)
    if right != -1:
        parent[right] = node_id
        direction[right] = "right"
        stack.append(right)

# Step 5: For each leaf, trace path back to root and collect required data
results = []

for leaf_id in leaf_ids:
    current = leaf_id
    path_info = []

    while parent[current] is not None:
        parent_id = parent[current]
        dir_taken = direction[current]

        path_info.append({
            "leaf_node": leaf_id,
            "traversed_node": parent_id,
            "direction_taken": dir_taken,
            "feature": features[parent_id],
            "samples_left": samples[children_left[parent_id]],
            "samples_right": samples[children_right[parent_id]],
            "psi": (-samples[children_right[parent_id]])/np.sqrt(samples[children_left[parent_id]]*samples[children_right[parent_id]]) if dir_taken == "left" else (samples[children_left[parent_id]])/np.sqrt(samples[children_left[parent_id]]*samples[children_right[parent_id]])
        })

        current = parent_id

    # Reverse to go from root to leaf
    results.extend(reversed(path_info))

# Step 6: Display
for row in results:
    print(f"Leaf Node {row['leaf_node']}:")
    print(f"  Traversed Node: {row['traversed_node']}")
    print(f"  Feature: {row['feature']}")
    print(f"  Direction Taken: {row['direction_taken']}")
    print(f"  Samples Left: {row['samples_left']}, Right: {row['samples_right']}")
    print(f"  Psi: {row['psi']:.4f}")
    print()


Leaf Node 2:
  Traversed Node: 0
  Feature: 1
  Direction Taken: left
  Samples Left: 14, Right: 50
  Psi: -1.8898

Leaf Node 2:
  Traversed Node: 1
  Feature: 0
  Direction Taken: left
  Samples Left: 8, Right: 6
  Psi: -0.8660

Leaf Node 3:
  Traversed Node: 0
  Feature: 1
  Direction Taken: left
  Samples Left: 14, Right: 50
  Psi: -1.8898

Leaf Node 3:
  Traversed Node: 1
  Feature: 0
  Direction Taken: right
  Samples Left: 8, Right: 6
  Psi: 1.1547

Leaf Node 5:
  Traversed Node: 0
  Feature: 1
  Direction Taken: right
  Samples Left: 14, Right: 50
  Psi: 0.5292

Leaf Node 5:
  Traversed Node: 4
  Feature: 1
  Direction Taken: left
  Samples Left: 49, Right: 1
  Psi: -0.1429

Leaf Node 6:
  Traversed Node: 0
  Feature: 1
  Direction Taken: right
  Samples Left: 14, Right: 50
  Psi: 0.5292

Leaf Node 6:
  Traversed Node: 4
  Feature: 1
  Direction Taken: right
  Samples Left: 49, Right: 1
  Psi: 7.0000



In [7]:
tree = rf.estimators_[0].tree_
n_nodes = tree.node_count
children_left = tree.children_left
children_right = tree.children_right
samples = tree.n_node_samples

# Step 3: Identify leaf and internal nodes
is_leaf = (children_left == -1) & (children_right == -1)
leaf_ids = [i for i in range(n_nodes) if is_leaf[i]]
internal_ids = [i for i in range(n_nodes) if not is_leaf[i]]

# Step 4: Build parent map and direction map
parent = {0: None}
direction = {}
stack = [0]

while stack:
    node_id = stack.pop()
    left = children_left[node_id]
    right = children_right[node_id]
    if left != -1:
        parent[left] = node_id
        direction[left] = "left"
        stack.append(left)
    if right != -1:
        parent[right] = node_id
        direction[right] = "right"
        stack.append(right)

# Step 5: Build the leaf-to-node psi matrix
psi_matrix = pd.DataFrame(0.0, index=leaf_ids, columns=internal_ids)

for leaf_id in leaf_ids:
    current = leaf_id
    traversed_nodes = []

    while parent[current] is not None:
        parent_id = parent[current]
        dir_taken = direction[current]

        # Get child sample sizes
        n_left = samples[children_left[parent_id]]
        n_right = samples[children_right[parent_id]]

        # Avoid division by zero
        denom = np.sqrt(n_left * n_right) if n_left > 0 and n_right > 0 else 1.0

        # Compute psi
        if dir_taken == "left":
            psi = (-n_right) / denom
        else:
            psi = (n_left) / denom

        # Set value in matrix
        psi_matrix.loc[leaf_id, parent_id] = psi

        current = parent_id
        
col_names = {
    node_id: f"node_{node_id} (X{features[node_id]})" if features[node_id] != -2 else f"node_{node_id} (leaf)"
    for node_id in psi_matrix.columns
}
psi_matrix.rename(columns=col_names, inplace=True)


# Step 6: Output the matrix
print(psi_matrix.round(3))


   node_0 (X1)  node_1 (X0)  node_4 (X1)
2       -1.890       -0.866        0.000
3       -1.890        1.155        0.000
5        0.529        0.000       -0.143
6        0.529        0.000        7.000


In [8]:
# add first and third columns
psi_test = psi_matrix.copy()
psi_test.iloc[:, 0] += psi_test.iloc[:, 2]
psi_test = psi_test.iloc[:, [1,0]]
psi_test.columns = ["X0", "X1"]

In [9]:
psi_test

,X0,X1
2,-0.866025,-1.889822
3,1.154701,-1.889822
5,0.000000,0.386293
6,0.000000,7.529150


In [10]:
X_leaf_ids = tree.apply(X_361254.astype(np.float32))

In [11]:
psi_feature_matrix = pd.DataFrame(
    [psi_test.loc[leaf_id].values for leaf_id in X_leaf_ids],
    columns=psi_test.columns,
    index=range(X_361254.shape[0])
)

# === Step 3: Done — inspect result ===
print(psi_feature_matrix.head())

         X0        X1
0  0.000000  0.386293
1  0.000000  0.386293
2 -0.866025 -1.889822
3  0.000000  0.386293
4  0.000000  0.386293


In [12]:
# fit linear regression to psi_feature_matrix, y
lr = LinearRegression()
lr.fit(psi_feature_matrix, y_361254.ravel())

# get feature importances
feature_importances = lr.coef_

In [13]:
feature_importances

array([4.42826847, 3.28347036])

In [14]:
# multiply psi_feature_matrix by feature importances
psi_feature_matrix_weighted = psi_feature_matrix * feature_importances
psi_feature_matrix_weighted.head()

,X0,X1
0,0.000000,1.268382
1,0.000000,1.268382
2,-3.834993,-6.205176
3,0.000000,1.268382
4,0.000000,1.268382


In [ ]:
# # add columns 1 and 3 since they both split on the same feature
# psi_feature_matrix_weighted["node_0 (X1)"] += psi_feature_matrix_weighted["node_4 (X1)"]
# psi_feature_matrix_weighted.drop(columns=["node_4 (X1)"], inplace=True)
# # rename columns for clarity
# psi_feature_matrix_weighted.rename(columns={"node_1 (X0)": "X0", "node_0 (X1)": "X1"}, inplace=True)
# psi_feature_matrix_weighted

In [15]:
# get unique rows in psi_feature_matrix_weighted
unique_rows = psi_feature_matrix_weighted.drop_duplicates()
unique_rows

,X0,X1
0,0.000000,1.268382
2,-3.834993,-6.205176
17,5.113324,-6.205176
29,0.000000,24.721742


In [ ]:
tree = rf.estimators_[0].tree_
n_nodes = tree.node_count
children_left = tree.children_left
children_right = tree.children_right
features = tree.feature
impurities = tree.impurity
samples = tree.n_node_samples

# Step 3: Identify leaf and internal nodes
is_leaf = (children_left == -1) & (children_right == -1)
leaf_ids = [i for i in range(n_nodes) if is_leaf[i]]
feature_indices = sorted(set(features[features >= 0]))

# Step 4: Build parent and direction maps
parent = {0: None}
direction = {}
stack = [0]

while stack:
    node_id = stack.pop()
    left = children_left[node_id]
    right = children_right[node_id]
    if left != -1:
        parent[left] = node_id
        direction[left] = "left"
        stack.append(left)
    if right != -1:
        parent[right] = node_id
        direction[right] = "right"
        stack.append(right)

# Step 5: Initialize feature contribution matrix
feature_matrix = pd.DataFrame(0.0, index=leaf_ids, columns=feature_indices)

# Step 6: For each leaf, trace back to root and accumulate impurity drops per feature
for leaf_id in leaf_ids:
    current = leaf_id
    while parent[current] is not None:
        parent_id = parent[current]
        dir_taken = direction[current]
        feat = features[parent_id]

        # Only consider actual splits
        if feat >= 0:
            # Take impurity of the child path we followed
            impurity_drop = impurities[parent_id] - impurities[current]
            feature_matrix.loc[leaf_id, feat] += impurity_drop

        current = parent_id

# Step 8: Show result
print(feature_matrix.round(4))


In [ ]:
rf_plus_mdi = LMDIPlus(rf_plus_inbag, evaluate_on="inbag")
lmdi= np.abs(rf_plus_mdi.get_lmdi_plus_scores(X=X_361254))#, y=y_361254)
threshold = 1e-10  # pick based on your noise scale
lmdi[np.abs(lmdi) < threshold] = 0
lmdi

In [ ]:
import numpy as np
import sklearn.ensemble

def compute_mdi_local_tree(tree, X, vimp):
    nsamples, nfeatures = X.shape

    impurity = tree.impurity
    threshold = tree.threshold
    children_left = tree.children_left
    children_right = tree.children_right
    features = tree.feature

    for i in range(nsamples):
        node = 0
        oldvimp = impurity[node]

        while children_left[node] != -1:
            ifeat = features[node]
            if X[i, ifeat] <= threshold[node]:
                node = children_left[node]
            else:
                node = children_right[node]
            newvimp = impurity[node]
            vimp[i, ifeat] += oldvimp - newvimp
            oldvimp = newvimp

def compute_mdi_local_ens(ens, X, verbose=0):
    nsamples, nfeatures = X.shape
    # vimp = np.zeros((nsamples, ens.n_features_), dtype='float64')
    vimp = np.zeros((nsamples, nfeatures), dtype='float64')

    for i, est in enumerate(ens.estimators_):
        if verbose > 0:
            print("o", end='', flush=True)
        compute_mdi_local_tree(est.tree_, X, vimp)

    if verbose > 0:
        print("")

    vimp /= ens.n_estimators
    return vimp

In [ ]:
lmdi_baseline = compute_mdi_local_ens(rf, X_361254)
lmdi_baseline